# 01 — Baseline Verification

This notebook verifies that the official CFSR backbone reproduces
the expected PSNR of **~32.33 dB** on Set5 (×4 super-resolution).

This is the essential first step — if the baseline numbers are wrong,
no delta measurement can be trusted.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
from src.models.cfsr import load_cfsr_model
from src.metrics.sr_metrics import evaluate_all_benchmarks

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Load Official CFSR Backbone

We use `strict=True` loading to ensure our architecture matches
the official checkpoint exactly.

In [ ]:
WEIGHTS = '../model_zoo/CFSR_x4.pth'
BENCH_DIR = '../datasets/benchmark'

model, param_count = load_cfsr_model(scale=4, weights_path=WEIGHTS, device=str(device))
print(f'Parameters: {param_count:,}')
print(f'Weights loaded with strict=True ✓')

## Evaluate on All 5 Benchmarks

In [ ]:
results = evaluate_all_benchmarks(
    model, scale=4, benchmark_dir=BENCH_DIR, device=str(device)
)

print(f"{'Dataset':<12} {'PSNR (dB)':<12} {'SSIM':<10}")
print('-' * 34)
for ds, r in results.items():
    print(f"{ds:<12} {r['psnr']:<12.4f} {r['ssim']:<10.4f}")

# Sanity check
set5_psnr = results['Set5']['psnr']
assert abs(set5_psnr - 32.33) < 0.05, f'Baseline WRONG: {set5_psnr}'
print(f'\n✅ Baseline verification PASSED (Set5 PSNR = {set5_psnr:.4f} dB)')